# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats.mstats import winsorize
from datetime import datetime

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
RAW_PATH = 'data/raw'
PROCESSED_PATH = 'data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [ ]:
# ====================== ЗАГРУЗКА ДАННЫХ ======================
orders = pd.read_csv(os.path.join(RAW_PATH, 'olist_orders_dataset.csv'),
                     parse_dates=['order_purchase_timestamp'])
order_items = pd.read_csv(
    os.path.join(RAW_PATH, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(RAW_PATH, 'olist_customers_dataset.csv'))
payments = pd.read_csv(
    os.path.join(RAW_PATH, 'olist_order_payments_dataset.csv'))
products = pd.read_csv(os.path.join(RAW_PATH, 'olist_products_dataset.csv'))

print(f"Orders: {orders.shape}")
print(
    f"Период данных: {orders['order_purchase_timestamp'].min()} → {orders['order_purchase_timestamp'].max()}"
)

In [4]:
# ====================== ОБЪЕДИНЕНИЕ ======================
df = orders.merge(order_items, on='order_id', how='left')
df = df.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id',
    how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(products[['product_id', 'product_category_name']],
              on='product_id',
              how='left')

# Оставляем только доставленные заказы (более чистый сигнал)
df = df[df['order_status'] == 'delivered'].copy()

print(f"После фильтра delivered: {df.shape}")
print(f"Уникальных клиентов: {df['customer_unique_id'].nunique():,}")

После фильтра delivered: (115038, 21)
Уникальных клиентов: 93,358


## Time-Split

Датасет Olist: ~2016-09 → 2018-08 (~24 месяца).

 cutoff = 2018-01-01
- Historical (признаки): всё до 2018-01-01
- Future (target): 2018-01-01 → конец данных (~8 месяцев)

Так мы получаем тысячи клиентов с future_ltv > 0.

In [5]:
CUTOFF = pd.Timestamp('2018-01-01')

historical = df[df['order_purchase_timestamp'] < CUTOFF].copy()
future = df[df['order_purchase_timestamp'] >= CUTOFF].copy()

print(
    f"Historical orders: {len(historical):,} | клиентов: {historical['customer_unique_id'].nunique():,}"
)
print(
    f"Future orders:     {len(future):,} | клиентов: {future['customer_unique_id'].nunique():,}"
)

# Target: future LTV (сумма payment_value после cutoff)
future_ltv = future.groupby(
    'customer_unique_id')['payment_value'].sum().reset_index()
future_ltv.columns = ['customer_unique_id', 'future_ltv']

print(f"\nКлиентов с future_ltv > 0: {(future_ltv['future_ltv'] > 0).sum():,}")
print(
    f"Средний future_ltv (среди >0): {future_ltv.loc[future_ltv['future_ltv'] > 0, 'future_ltv'].mean():.2f}"
)

Historical orders: 52,482 | клиентов: 42,395
Future orders:     62,556 | клиентов: 51,612

Клиентов с future_ltv > 0: 51,612
Средний future_ltv (среди >0): 210.05


## Feature Engineering (только на historical)

In [6]:
# ====================== RFM + дополнительные фичи ======================
rfm = historical.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (CUTOFF - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum'),
    avg_order_value=('payment_value', 'mean'),
    category_diversity=('product_category_name', 'nunique'),
    n_items=('order_item_id', 'count'),
    first_purchase_day=(
        'order_purchase_timestamp', lambda x:
        (x.min() - historical['order_purchase_timestamp'].min()).days),
    last_purchase_day=(
        'order_purchase_timestamp', lambda x:
        (x.max() - historical['order_purchase_timestamp'].min()).days),
).reset_index()


# Средний интервал между заказами
def avg_days_between(x):
    if len(x) < 2:
        return 0.0
    diffs = x.sort_values().diff().dt.days.dropna()
    return diffs.mean() if len(diffs) > 0 else 0.0


avg_interval = historical.groupby(
    'customer_unique_id')['order_purchase_timestamp'].apply(avg_days_between)
rfm = rfm.merge(avg_interval.rename('avg_days_between_orders'),
                on='customer_unique_id',
                how='left')

print(rfm.shape)
rfm.head()

(42395, 10)


,customer_unique_id,recency,frequency,monetary,avg_order_value,category_diversity,n_items,first_purchase_day,last_purchase_day,avg_days_between_orders
0,0000f46a3911fa3c0805444483337064,296,1,86.22,86.22,1,1,176,176,0.00
1,0000f6ccb0745a6a4b88665a16c9f078,80,1,43.62,43.62,1,1,392,392,0.00
2,0004aac84e0df4da2b147fca70cf8255,47,1,196.89,196.89,1,1,425,425,0.00
3,0005e1862207bf6ccc02e4228effd9a0,302,1,150.12,150.12,1,1,170,170,0.00
4,0006fdc98a402fceb4eb0ee528f6a8d4,166,1,29.00,29.00,1,1,305,305,0.00


In [7]:
# ====================== ОБРАБОТКА ВЫБРОСОВ ======================
rfm['monetary'] = winsorize(rfm['monetary'], limits=[0, 0.01])
rfm['avg_order_value'] = winsorize(rfm['avg_order_value'], limits=[0, 0.01])

In [8]:
# ====================== PAYMENT TYPE (исправленный merge) ======================
# Берём самый частый способ оплаты клиента
payment_main = (historical.groupby(['customer_unique_id', 'payment_type'
                                    ]).size().reset_index(name='count'))
payment_main = payment_main.loc[payment_main.groupby('customer_unique_id')
                                ['count'].idxmax()]

payment_ohe = pd.get_dummies(payment_main['payment_type'],
                             prefix='payment',
                             dtype=int)
payment_ohe['customer_unique_id'] = payment_main['customer_unique_id'].values

rfm = rfm.merge(payment_ohe, on='customer_unique_id', how='left')
payment_cols = [c for c in rfm.columns if c.startswith('payment_')]
rfm[payment_cols] = rfm[payment_cols].fillna(0)

print("Payment columns:", payment_cols)

Payment columns: ['payment_boleto', 'payment_credit_card', 'payment_debit_card', 'payment_voucher']


In [9]:
# ====================== GEO (штат) ======================
state = historical.groupby(
    'customer_unique_id')['customer_state'].first().reset_index()
state_ohe = pd.get_dummies(state['customer_state'], prefix='state', dtype=int)
state_ohe['customer_unique_id'] = state['customer_unique_id'].values

rfm = rfm.merge(state_ohe, on='customer_unique_id', how='left')
state_cols = [c for c in rfm.columns if c.startswith('state_')]
rfm[state_cols] = rfm[state_cols].fillna(0)

print(f"Добавлено штатов: {len(state_cols)}")

Добавлено штатов: 27


In [10]:
# ====================== INTERACTION FEATURES ======================
rfm['freq_x_diversity'] = rfm['frequency'] * rfm['category_diversity']
rfm['monetary_per_order'] = rfm['monetary'] / rfm['frequency'].clip(lower=1)
rfm['items_per_order'] = rfm['n_items'] / rfm['frequency'].clip(lower=1)

In [11]:
# ====================== TARGET ======================
rfm = rfm.merge(future_ltv, on='customer_unique_id', how='left')
rfm['future_ltv'] = rfm['future_ltv'].fillna(0)
rfm['will_return'] = (rfm['future_ltv'] > 0).astype(int)

print(f"Всего клиентов в historical: {len(rfm):,}")
print(
    f"will_return = 1: {rfm['will_return'].sum():,} ({rfm['will_return'].mean():.2%})"
)
print(f"Средний future_ltv (все): {rfm['future_ltv'].mean():.2f}")
print(
    f"Средний future_ltv (только returners): {rfm.loc[rfm['will_return']==1, 'future_ltv'].mean():.2f}"
)

Всего клиентов в historical: 42,395
will_return = 1: 649 (1.53%)
Средний future_ltv (все): 3.52
Средний future_ltv (только returners): 229.83


In [12]:
# ====================== ФИНАЛЬНЫЙ НАБОР ПРИЗНАКОВ ======================
# Убираем recency (сильный leakage-риск) и служебные колонки
drop_cols = ['customer_unique_id', 'recency', 'future_ltv', 'will_return']
feature_cols = [c for c in rfm.columns if c not in drop_cols]

final_df = rfm[['customer_unique_id'] + feature_cols +
               ['future_ltv', 'will_return']].copy()

print("Финальные признаки:")
print(final_df.dtypes)
print(f"\nShape: {final_df.shape}")
final_df.head()

Финальные признаки:
customer_unique_id          object
frequency                    int64
monetary                   float64
avg_order_value            float64
category_diversity           int64
n_items                      int64
first_purchase_day           int64
last_purchase_day            int64
avg_days_between_orders    float64
payment_boleto             float64
payment_credit_card        float64
payment_debit_card         float64
payment_voucher            float64
state_AC                     int64
state_AL                     int64
state_AM                     int64
state_AP                     int64
state_BA                     int64
state_CE                     int64
state_DF                     int64
state_ES                     int64
state_GO                     int64
state_MA                     int64
state_MG                     int64
state_MS                     int64
state_MT                     int64
state_PA                     int64
state_PB                     int64


,customer_unique_id,frequency,monetary,avg_order_value,category_diversity,n_items,first_purchase_day,last_purchase_day,avg_days_between_orders,payment_boleto,payment_credit_card,payment_debit_card,payment_voucher,state_AC,state_AL,state_AM,state_AP,state_BA,state_CE,state_DF,state_ES,state_GO,state_MA,state_MG,state_MS,state_MT,state_PA,state_PB,state_PE,state_PI,state_PR,state_RJ,state_RN,state_RO,state_RR,state_RS,state_SC,state_SE,state_SP,state_TO,freq_x_diversity,monetary_per_order,items_per_order,future_ltv,will_return
0,0000f46a3911fa3c0805444483337064,1,86.22,86.22,1,1,176,176,0.00,0.00,1.00,0.00,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,86.22,1.00,0.00,0
1,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,1,1,392,392,0.00,0.00,1.00,0.00,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,43.62,1.00,0.00,0
2,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,1,1,425,425,0.00,0.00,1.00,0.00,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,196.89,1.00,0.00,0
3,0005e1862207bf6ccc02e4228effd9a0,1,150.12,150.12,1,1,170,170,0.00,0.00,1.00,0.00,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,150.12,1.00,0.00,0
4,0006fdc98a402fceb4eb0ee528f6a8d4,1,29.00,29.00,1,1,305,305,0.00,0.00,1.00,0.00,0.00,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,29.00,1.00,0.00,0


In [14]:
# ====================== СОХРАНЕНИЕ ======================
out_path = os.path.join(PROCESSED_PATH, 'features_.csv')
final_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

# Краткая статистика
print("\n=== Summary ===")
print(f"Клиентов: {len(final_df):,}")
print(
    f"Позитивов (will_return): {final_df['will_return'].sum():,} ({final_df['will_return'].mean():.2%})"
)
print(f"Признаков: {len(feature_cols)}")

Saved → data/processed/features_.csv

=== Summary ===
Клиентов: 42,395
Позитивов (will_return): 649 (1.53%)
Признаков: 42
